# Medical-document RAG learning template

This notebook demonstrates a retrieval-augmented generation workflow:
1. Extract text from trusted medical documents through an API.
2. Split the text into searchable chunks.
3. Create embeddings and retrieve relevant chunks.
4. Generate an answer grounded only in the retrieved sources.

> Safety: this is an educational prototype. It must not diagnose, prescribe, replace a clinician, or be used as an emergency service. Keep source citations in every answer and direct urgent or personal medical questions to a qualified professional.

## 1. Install dependencies

Run this cell once in a new environment.

In [ ]:
%pip install requests python-dotenv openai numpy

## 2. Imports and configuration

Fill in the values marked `TODO`. Do not hard-code private API keys in a notebook that will be shared.

In [ ]:
import os
import requests
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# TODO 1: put your keys in a .env file, not directly in this notebook.
DOCUMENT_API_KEY = os.getenv("DOCUMENT_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# TODO 2: replace this with the document-extraction API endpoint you selected.
DOCUMENT_API_URL = "https://YOUR-DOCUMENT-API.example.com/extract"

# TODO 3: choose models available in your account.
EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "YOUR_CHAT_MODEL"

client = OpenAI(api_key=OPENAI_API_KEY)
assert DOCUMENT_API_KEY, "Fill DOCUMENT_API_KEY in .env"
assert OPENAI_API_KEY, "Fill OPENAI_API_KEY in .env"

## 3. Choose trusted source documents

Use documents you are allowed to process. Prefer official clinical guidelines, government health agencies, hospitals, systematic reviews, or approved institutional materials. Record the title, publisher and publication date for citation.

In [ ]:
DOCUMENTS = [
    {
        "title": "TODO: document title",
        "publisher": "TODO: publisher or health authority",
        "date": "TODO: publication or update date",
        "url": "TODO: document URL or API source identifier",
        "api_input": "TODO: URL, file ID, or payload expected by your extraction API"
    }
]

## 4. Extract document text through an API

The request format differs between providers. Replace the placeholder payload and response field after reading your provider's documentation.

In [ ]:
def extract_document(doc):
    headers = {
        "Authorization": f"Bearer {DOCUMENT_API_KEY}",
        "Content-Type": "application/json",
    }

    # TODO 4: change this payload to match your document API.
    payload = {"source": doc["api_input"]}
    response = requests.post(DOCUMENT_API_URL, json=payload, headers=headers, timeout=60)
    response.raise_for_status()
    data = response.json()

    # TODO 5: change this field to the API's extracted-text field.
    text = data["text"]
    return {**doc, "text": text}

extracted_documents = [extract_document(doc) for doc in DOCUMENTS]
print(extracted_documents[0]["text"][:500])

## 5. Chunk the documents

Chunks should be large enough to preserve meaning but small enough to retrieve precisely. Keep document metadata attached to every chunk so the answer can cite its source.

In [ ]:
def make_chunks(documents, chunk_size=900, overlap=150):
    chunks = []
    for doc in documents:
        text = doc["text"]
        start = 0
        chunk_id = 0
        while start < len(text):
            end = min(start + chunk_size, len(text))
            chunks.append({
                "chunk_id": f"{doc['title']}-{chunk_id}",
                "text": text[start:end],
                "title": doc["title"],
                "publisher": doc["publisher"],
                "date": doc["date"],
                "url": doc["url"],
            })
            if end == len(text):
                break
            start = end - overlap
            chunk_id += 1
    return chunks

chunks = make_chunks(extracted_documents)
print(f"Created {len(chunks)} chunks")

## 6. Create embeddings and retrieve relevant chunks

This simple in-memory index is suitable for learning. A production system would normally use a managed vector database and access controls.

In [ ]:
def embed_texts(texts):
    result = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return np.array([item.embedding for item in result.data], dtype=np.float32)

def normalise(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.clip(norms, 1e-12, None)

chunk_vectors = normalise(embed_texts([c["text"] for c in chunks]))

def retrieve(question, top_k=5):
    question_vector = normalise(embed_texts([question]))[0]
    scores = chunk_vectors @ question_vector
    best_indices = np.argsort(scores)[::-1][:top_k]
    return [{**chunks[i], "score": float(scores[i])} for i in best_indices]

## 7. Generate a grounded medical-information answer

The prompt requires the model to use only retrieved content, identify uncertainty, cite sources and avoid diagnosis or treatment instructions.

In [ ]:
def answer_question(question, top_k=5):
    retrieved = retrieve(question, top_k=top_k)
    context = "\n\n".join(
        f"[Source {i+1}] {r['title']} | {r['publisher']} | {r['date']} | {r['url']}\n{r['text']}"
        for i, r in enumerate(retrieved)
    )

    system_prompt = """
You are a medical-document question-answering assistant.
Use only the supplied sources. If the sources do not answer the question, say so.
Do not diagnose, prescribe, interpret a personal case, or provide emergency instructions.
Use cautious language, distinguish evidence from uncertainty, and cite source numbers.
Remind the reader to consult a qualified healthcare professional for personal decisions.
"""

    user_prompt = f"Question: {question}\n\nSources:\n{context}"
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content, retrieved

## 8. Try one question

Use a question answerable from the documents. Do not enter identifiable patient information.

In [ ]:
QUESTION = "TODO: write a general question answerable from the selected documents"
answer, sources = answer_question(QUESTION)
print(answer)
print("\nRetrieved sources:")
for source in sources:
    print(f"- {source['title']} ({source['publisher']}, {source['date']}) | score={source['score']:.3f}")

## 9. Evaluation checklist

Before using the prototype, test it with questions whose answers are known from the documents.

- Does retrieval return the correct source section?
- Does the answer stay within the supplied documents?
- Are citations present and accurate?
- Does the system admit when evidence is missing?
- Does it avoid diagnosis and personalised treatment advice?
- Does it protect private or identifiable information?
- Are document dates and publisher authority recorded?